# 03 - Feature Engineering

## Objective

The objective of this notebook is to transform the raw loan application
data into a clean and modelling-ready dataset.

The feature engineering process is guided by the findings from the
exploratory data analysis. Particular attention is given to:

- implausible and extreme observations;
- consistency between applicant age and employment experience;
- highly correlated variables;
- potential data leakage;
- financially meaningful derived variables;
- categorical variable encoding; and
- preparation of the predictor matrix and target variable.

The aim is to improve the quality and interpretability of the features
without introducing information that would not be available when a loan
application is being evaluated.

# Import libraries

In [2]:
import pandas as pd
import numpy as np

# Loading data for further preprocessing

In [3]:
loan_data=pd.read_csv("/workspaces/loan-approval-prediction/data/raw/loan_data_new.csv")

In [4]:
print("Shape:", loan_data.shape)
print("\nColumns:")
print(loan_data.columns.tolist())

Shape: (45000, 14)

Columns:
['Age', 'Gender', 'Education', 'Person Income', 'Employee Experience', 'Home Onwership', 'Loan Amount', 'Loan Intent', 'Loan interest Rate', 'Loan percentage', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan Status']


Confirming if there's no missing observations

In [5]:
loan_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  45000 non-null  int64  
 1   Gender               45000 non-null  str    
 2   Education            45000 non-null  str    
 3   Person Income        45000 non-null  int64  
 4   Employee Experience  45000 non-null  int64  
 5   Home Onwership       45000 non-null  str    
 6   Loan Amount          45000 non-null  int64  
 7   Loan Intent          45000 non-null  str    
 8   Loan interest Rate   45000 non-null  float64
 9   Loan percentage      45000 non-null  float64
 10  Credit History       45000 non-null  int64  
 11  Credit Score         45000 non-null  int64  
 12  Previous Loan        45000 non-null  str    
 13  Loan Status          45000 non-null  int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 6.2 MB


# Making a copy of my data

In [6]:
new_df = loan_data.copy()

print("Original dataset:", loan_data.shape)
print("Working dataset:", new_df.shape)

Original dataset: (45000, 14)
Working dataset: (45000, 14)


Defining the target variable:

In [7]:
new_df["Loan Status"].value_counts()

Loan Status
0    35000
1    10000
Name: count, dtype: int64

In [8]:
new_df["Loan Status"].value_counts(normalize=True) * 100

Loan Status
0    77.777778
1    22.222222
Name: proportion, dtype: float64

In [9]:
target = "Loan Status"

In [10]:
X = new_df.drop(columns=[target])
y = new_df[target]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (45000, 13)
Target: (45000,)


#  Investigate Implausible and Extreme Values (Outliers)

The exploratory data analysis identified several variables with unusually
large values. These observations are investigated before any transformation
or removal is performed.

The variables of particular interest are:

- Age
- Employee Experience
- Person Income

The purpose of this investigation is to distinguish genuine extreme
observations from potentially invalid or erroneous records.

In [11]:
new_df[["Age", "Employee Experience", "Person Income"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,45000.0,27.764178,6.045108,20.0,24.0,26.0,30.00,144.0
Employee Experience,45000.0,5.410333,6.063532,0.0,1.0,4.0,8.00,125.0
Person Income,45000.0,80319.053222,80422.498632,8000.0,47204.0,67048.0,95789.25,7200766.0


inspecting the largest observation on Age variable

In [12]:
new_df.nlargest(10, "Age")[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Credit History",
    "Credit Score",
    "Loan Amount",
    "Loan Status"
]]

,Age,Employee Experience,Person Income,Credit History,Credit Score,Loan Amount,Loan Status
81,144,125,300616,3,789,4800,0
183,144,121,241424,2,807,6000,0
32297,144,124,7200766,25,850,5000,0
575,123,101,97140,3,805,20400,0
747,123,100,94723,4,714,20000,0
37930,116,93,5545545,24,708,3823,0
38113,109,85,5556399,22,792,6195,0
32416,94,76,29738,27,773,6500,0
32506,84,61,114705,24,784,10000,0
32422,80,62,77894,25,673,6800,0


inspecting the largest observation on Employee Exprerience variable

In [13]:
new_df.nlargest(10, "Employee Experience")[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Credit History",
    "Credit Score",
    "Loan Amount",
    "Loan Status"
]]

,Age,Employee Experience,Person Income,Credit History,Credit Score,Loan Amount,Loan Status
81,144,125,300616,3,789,4800,0
32297,144,124,7200766,25,850,5000,0
183,144,121,241424,2,807,6000,0
575,123,101,97140,3,805,20400,0
747,123,100,94723,4,714,20000,0
37930,116,93,5545545,24,708,3823,0
38113,109,85,5556399,22,792,6195,0
32416,94,76,29738,27,773,6500,0
32422,80,62,77894,25,673,6800,0
32506,84,61,114705,24,784,10000,0


inspecting the largest observation on Personal Income variable

In [14]:
new_df.nlargest(10, "Person Income")[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Loan Amount",
    "Loan percentage",
    "Loan Status"
]]

,Age,Employee Experience,Person Income,Loan Amount,Loan percentage,Loan Status
32297,144,124,7200766,5000,0.00,0
38113,109,85,5556399,6195,0.00,0
37930,116,93,5545545,3823,0.00,0
30049,42,16,2448661,8450,0.00,0
32546,60,37,2280980,1500,0.00,0
32497,63,43,2139143,12025,0.01,0
37175,42,19,2012954,9121,0.00,0
41288,46,22,1741243,12011,0.01,0
31924,44,20,1728974,6400,0.00,0
35850,47,22,1661567,6545,0.00,0


Checking logical consistency

In [16]:
new_df[new_df["Employee Experience"] > new_df["Age"]][[
    "Age",
    "Employee Experience",
    "Person Income",
    "Credit History",
    "Credit Score",
    "Loan Status"
]].head()

,Age,Employee Experience,Person Income,Credit History,Credit Score,Loan Status


In [17]:
(new_df["Employee Experience"] > new_df["Age"]).sum()

np.int64(0)

Quantifying the suspicious observations

Age

In [18]:
age_check = new_df[new_df["Age"] > 80]

print("Applicants with Age > 80:", len(age_check))
print("Percentage:", len(age_check) / len(new_df) * 100)

age_check[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Credit History",
    "Credit Score",
    "Loan Status"
]].sort_values("Age", ascending=False).head(20)

Applicants with Age > 80: 9
Percentage: 0.02


,Age,Employee Experience,Person Income,Credit History,Credit Score,Loan Status
81,144,125,300616,3,789,0
183,144,121,241424,2,807,0
32297,144,124,7200766,25,850,0
575,123,101,97140,3,805,0
747,123,100,94723,4,714,0
37930,116,93,5545545,24,708,0
38113,109,85,5556399,22,792,0
32416,94,76,29738,27,773,0
32506,84,61,114705,24,784,0


Employee Experience

In [19]:
experience_check = new_df[new_df["Employee Experience"] > 50]

print("Applicants with Experience > 50:", len(experience_check))
print("Percentage:", len(experience_check) / len(new_df) * 100)

experience_check[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Credit History",
    "Credit Score",
    "Loan Status"
]].sort_values("Employee Experience", ascending=False).head(20)

Applicants with Experience > 50: 12
Percentage: 0.02666666666666667


,Age,Employee Experience,Person Income,Credit History,Credit Score,Loan Status
81,144,125,300616,3,789,0
32297,144,124,7200766,25,850,0
183,144,121,241424,2,807,0
575,123,101,97140,3,805,0
747,123,100,94723,4,714,0
37930,116,93,5545545,24,708,0
38113,109,85,5556399,22,792,0
32416,94,76,29738,27,773,0
32422,80,62,77894,25,673,0
32506,84,61,114705,24,784,0


Income

In [20]:
income_check = new_df[new_df["Person Income"] > 500000]

print("Applicants with Income > 500,000:", len(income_check))
print("Percentage:", len(income_check) / len(new_df) * 100)

income_check[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Loan Amount",
    "Loan percentage",
    "Loan Status"
]].sort_values("Person Income", ascending=False).head(20)

Applicants with Income > 500,000: 104
Percentage: 0.2311111111111111


,Age,Employee Experience,Person Income,Loan Amount,Loan percentage,Loan Status
32297,144,124,7200766,5000,0.00,0
38113,109,85,5556399,6195,0.00,0
37930,116,93,5545545,3823,0.00,0
30049,42,16,2448661,8450,0.00,0
32546,60,37,2280980,1500,0.00,0
32497,63,43,2139143,12025,0.01,0
37175,42,19,2012954,9121,0.00,0
41288,46,22,1741243,12011,0.01,0
31924,44,20,1728974,6400,0.00,0
35850,47,22,1661567,6545,0.00,0


Age and Experience consistency

In [21]:
inconsistent_experience = new_df[
    new_df["Employee Experience"] > new_df["Age"]
]

print("Experience greater than age:", len(inconsistent_experience))
print("Percentage:", len(inconsistent_experience) / len(new_df) * 100)

Experience greater than age: 0
Percentage: 0.0


In [22]:
inconsistent_experience[[
    "Age",
    "Employee Experience",
    "Person Income",
    "Credit History",
    "Credit Score",
    "Loan Status"
]].sort_values(
    "Employee Experience",
    ascending=False
).head(20)

,Age,Employee Experience,Person Income,Credit History,Credit Score,Loan Status


Quantifying exactly how many observations would be removed if two clear quality rules are applied:

In [23]:
invalid_records = new_df[
    (new_df["Age"] > 80) |
    (new_df["Employee Experience"] > 50)
]

print("Records flagged for removal:", len(invalid_records))
print(
    "Percentage of dataset:",
    round(len(invalid_records) / len(new_df) * 100, 4)
)

Records flagged for removal: 12
Percentage of dataset: 0.0267


In [24]:
both_invalid = new_df[
    (new_df["Age"] > 80) &
    (new_df["Employee Experience"] > 50)
]

print("Records violating both rules:", len(both_invalid))

Records violating both rules: 9


Removing invalid observations

In [25]:
# Defining invalid observations
invalid_mask = (
    (new_df["Age"] > 80) |
    (new_df["Employee Experience"] > 50)
)

# Removing invalid observations
df_clean = new_df.loc[~invalid_mask].copy()

print("Original dataset shape:", new_df.shape)
print("Cleaned dataset shape:", df_clean.shape)
print("Records removed:", len(new_df) - len(df_clean))

Original dataset shape: (45000, 14)
Cleaned dataset shape: (44988, 14)
Records removed: 12


Verify the cleaning

In [26]:
print("Maximum age:", df_clean["Age"].max())
print("Maximum employee experience:", df_clean["Employee Experience"].max())

print("\nRemaining invalid ages:")
print((df_clean["Age"] > 80).sum())

print("\nRemaining invalid experience values:")
print((df_clean["Employee Experience"] > 50).sum())

Maximum age: 73
Maximum employee experience: 50

Remaining invalid ages:
0

Remaining invalid experience values:
0


### Handling Implausible Observations

The exploratory analysis identified implausible observations in the 'Age' and
'Employee Experience' variables. Nine applicants had an age greater than 80
years, while 12 applicants had more than 50 years of employee experience.
Because nine observations violated both conditions, these criteria identified
12 unique observations for removal.

These observations represented only 0.0267% of the original 45,000 records.
The extreme values were therefore treated as data-quality anomalies rather
than genuine observations. The affected records were removed from the
modelling dataset rather than capped or winsorised, as retaining such values
could introduce unrealistic applicant profiles into the predictive models.

After removal, the dataset contained 44,988 observations.

# Investigating the financial variables

In [27]:
# Calculating loan-to-income ratio independently
calculated_lti = (
    df_clean["Loan Amount"] /
    df_clean["Person Income"]
)

# Compared with the supplied Loan percentage
comparison = pd.DataFrame({
    "Loan Amount": df_clean["Loan Amount"],
    "Person Income": df_clean["Person Income"],
    "Supplied Loan percentage": df_clean["Loan percentage"],
    "Calculated Loan-to-Income": calculated_lti
})

comparison["Difference"] = (
    comparison["Supplied Loan percentage"] -
    comparison["Calculated Loan-to-Income"]
)

comparison.head(10)

,Loan Amount,Person Income,Supplied Loan percentage,Calculated Loan-to-Income,Difference
0,35000,71948,0.49,0.486462,0.003538
1,1000,12282,0.08,0.081420,-0.001420
2,5500,12438,0.44,0.442193,-0.002193
3,35000,79753,0.44,0.438855,0.001145
4,35000,66135,0.53,0.529221,0.000779
5,2500,12951,0.19,0.193035,-0.003035
6,35000,93471,0.37,0.374448,-0.004448
7,35000,95550,0.37,0.366300,0.003700
8,35000,100684,0.35,0.347622,0.002378
9,1600,12739,0.13,0.125599,0.004401


In [28]:
print("Maximum absolute difference:",
      comparison["Difference"].abs().max())

print("Mean absolute difference:",
      comparison["Difference"].abs().mean())

Maximum absolute difference: 0.00503013444598982
Mean absolute difference: 0.0025249191353030353


# Creating my own precise Loan-to-Income Ratio and dropping the supplied Loan percentage

Whereby: Loan-to-Income Ratio (LTI) measures the size of the requested loan relative to the applicant's income and provides an indication of the financial burden associated with the requested borrowing.

In [29]:
# Create precise Loan-to-Income ratio
df_clean["Loan_to_Income"] = (
    df_clean["Loan Amount"] /
    df_clean["Person Income"]
)

# Remove the rounded duplicate feature
df_clean = df_clean.drop(columns=["Loan percentage"])

print("Updated dataset shape:", df_clean.shape)

df_clean[[
    "Person Income",
    "Loan Amount",
    "Loan_to_Income"
]].head()

Updated dataset shape: (44988, 14)


,Person Income,Loan Amount,Loan_to_Income
0,71948,35000,0.486462
1,12282,1000,0.081420
2,12438,5500,0.442193
3,79753,35000,0.438855
4,66135,35000,0.529221


# investigating Loan interest Rate

EDA showed that Loan interest Rate had a relatively strong relationship with Loan Status (correlation ≈ 0.33).

In [30]:
df_clean.groupby("Loan Status")["Loan interest Rate"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max"
])

,count,mean,median,std,min,max
Loan Status,,,,,,
0,34988,10.477835,10.85,2.731718,5.42,20.0
1,10000,12.856794,12.98,3.068512,5.42,20.0


In [31]:
df_clean.groupby("Loan Status")["Loan interest Rate"].describe()

,count,mean,std,min,25%,50%,75%,max
Loan Status,,,,,,,,
0,34988.0,10.477835,2.731718,5.42,7.90,10.85,12.29,20.0
1,10000.0,12.856794,3.068512,5.42,10.99,12.98,15.23,20.0


In [32]:
df_clean[["Loan Status", "Loan interest Rate"]].corr()

,Loan Status,Loan interest Rate
Loan Status,1.000000,0.332031
Loan interest Rate,0.332031,1.000000


Would the interest rate be known at the time the loan approval decision is made?

# inspect Previous Loan

In [33]:
print("Previous Loan values:")
print(df_clean["Previous Loan"].value_counts(dropna=False))

print("\nUnique values:")
print(df_clean["Previous Loan"].unique())

Previous Loan values:
Previous Loan
Yes    22855
No     22133
Name: count, dtype: int64

Unique values:
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


inspect the other categorical variables:

In [34]:
categorical_columns = [
    "Gender",
    "Education",
    "Home Onwership",
    "Loan Intent",
    "Previous Loan"
]

for col in categorical_columns:
    print(f"\n{'='*50}")
    print(col)
    print(df_clean[col].value_counts(dropna=False))


Gender
Gender
male      24832
female    20156
Name: count, dtype: int64

Education
Education
Bachelor       13395
Associate      12025
High School    11967
Master          6980
Doctorate        621
Name: count, dtype: int64

Home Onwership
Home Onwership
RENT        23436
MORTGAGE    18484
OWN          2951
OTHER         117
Name: count, dtype: int64

Loan Intent
Loan Intent
EDUCATION            9151
MEDICAL              8544
VENTURE              7815
PERSONAL             7550
DEBTCONSOLIDATION    7145
HOMEIMPROVEMENT      4783
Name: count, dtype: int64

Previous Loan
Previous Loan
Yes    22855
No     22133
Name: count, dtype: int64


In [35]:
# Correct column name spelling
df_clean = df_clean.rename(
    columns={"Home Onwership": "Home Ownership"}
)

print(df_clean.columns.tolist())

['Age', 'Gender', 'Education', 'Person Income', 'Employee Experience', 'Home Ownership', 'Loan Amount', 'Loan Intent', 'Loan interest Rate', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan Status', 'Loan_to_Income']


In [36]:
print("Final feature-engineering dataset shape:", df_clean.shape)

print("\nColumns:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"{i}. {col}")

Final feature-engineering dataset shape: (44988, 14)

Columns:
1. Age
2. Gender
3. Education
4. Person Income
5. Employee Experience
6. Home Ownership
7. Loan Amount
8. Loan Intent
9. Loan interest Rate
10. Credit History
11. Credit Score
12. Previous Loan
13. Loan Status
14. Loan_to_Income


In [37]:
print("\nTarget distribution:")
print(df_clean["Loan Status"].value_counts())

print("\nTarget percentages:")
print(
    df_clean["Loan Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Target distribution:
Loan Status
0    34988
1    10000
Name: count, dtype: int64

Target percentages:
Loan Status
0    77.77
1    22.23
Name: proportion, dtype: float64


Checking income skewness

In [38]:
# Income distribution summary
df_clean["Person Income"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    4.498800e+04
mean     7.990907e+04
std      6.332489e+04
min      8.000000e+03
1%       1.787535e+04
5%       2.836310e+04
25%      4.719425e+04
50%      6.704550e+04
75%      9.577900e+04
95%      1.667018e+05
99%      2.712701e+05
max      2.448661e+06
Name: Person Income, dtype: float64

In [39]:
# Log-transform income to reduce extreme right skew
df_clean["Log_Person_Income"] = np.log1p(df_clean["Person Income"])

df_clean[
    ["Person Income", "Log_Person_Income"]
].describe()

,Person Income,Log_Person_Income
count,4.498800e+04,44988.000000
mean,7.990907e+04,11.122080
std,6.332489e+04,0.556264
min,8.000000e+03,8.987322
25%,4.719425e+04,10.762049
50%,6.704550e+04,11.113142
75%,9.577900e+04,11.469809
max,2.448661e+06,14.711052


Comparing the skewness:

In [40]:
print("Original income skewness:",
      df_clean["Person Income"].skew())

print("Log income skewness:",
      df_clean["Log_Person_Income"].skew())

Original income skewness: 9.694780162911552
Log income skewness: 0.19171511694784846


### Income Transformation

The "Person Income" variable exhibited substantial positive skewness, with a
skewness coefficient of 9.695 and a maximum observed income of approximately
R2.45 million after data-quality cleaning. Since high income values may
represent legitimate applicants, these observations were not removed or
capped.

A logarithmic transformation was therefore applied:

(Log_Person_Income)_i = log(1+ {Person Income}_i).

The transformation reduced the skewness from 9.695 to 0.192, producing a
substantially more symmetric distribution while retaining all valid income
observations. The transformed variable is retained as a candidate modelling
feature, particularly for models sensitive to feature scale and skewness.

In [41]:
print("Dataset shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:")
print(df_clean.duplicated().sum())

Dataset shape: (44988, 15)

Columns:
['Age', 'Gender', 'Education', 'Person Income', 'Employee Experience', 'Home Ownership', 'Loan Amount', 'Loan Intent', 'Loan interest Rate', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan Status', 'Loan_to_Income', 'Log_Person_Income']

Missing values:
Age                    0
Gender                 0
Education              0
Person Income          0
Employee Experience    0
Home Ownership         0
Loan Amount            0
Loan Intent            0
Loan interest Rate     0
Credit History         0
Credit Score           0
Previous Loan          0
Loan Status            0
Loan_to_Income         0
Log_Person_Income      0
dtype: int64

Duplicate rows:
0


In [42]:
numeric_features = df_clean.select_dtypes(
    include=["int64", "float64"]
).columns

df_clean[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,44988.0,27.742354,5.881527,20.000000,24.000000,26.000000,30.000000,7.300000e+01
Person Income,44988.0,79909.065706,63324.894109,8000.000000,47194.250000,67045.500000,95779.000000,2.448661e+06
Employee Experience,44988.0,5.388148,5.896067,0.000000,1.000000,4.000000,8.000000,5.000000e+01
Loan Amount,44988.0,9583.323820,6314.994983,500.000000,5000.000000,8000.000000,12237.250000,3.500000e+04
Loan interest Rate,44988.0,11.006634,2.979042,5.420000,8.590000,11.010000,12.990000,2.000000e+01
Credit History,44988.0,5.864408,3.872007,2.000000,3.000000,4.000000,8.000000,3.000000e+01
Credit Score,44988.0,632.573308,50.389790,390.000000,601.000000,640.000000,670.000000,7.840000e+02
Loan Status,44988.0,0.222281,0.415784,0.000000,0.000000,0.000000,0.000000,1.000000e+00
Loan_to_Income,44988.0,0.139761,0.087163,0.000658,0.073565,0.121668,0.188144,6.641860e-01
Log_Person_Income,44988.0,11.122080,0.556264,8.987322,10.762049,11.113142,11.469809,1.471105e+01


In [43]:
print("Numeric feature correlations with Loan Status:")

correlations = (
    df_clean[numeric_features]
    .corr()["Loan Status"]
    .sort_values(ascending=False)
)

print(correlations)

Numeric feature correlations with Loan Status:
Loan Status            1.000000
Loan_to_Income         0.385010
Loan interest Rate     0.332031
Loan Amount            0.107717
Credit Score          -0.007279
Credit History        -0.014457
Employee Experience   -0.019055
Age                   -0.020093
Person Income         -0.169044
Log_Person_Income     -0.283783
Name: Loan Status, dtype: float64


In [44]:
# Remove potential target-leakage variable
df_clean = df_clean.drop(columns=["Loan interest Rate"])

print("Dataset shape after leakage removal:", df_clean.shape)
print("\nColumns:")
print(df_clean.columns.tolist())

Dataset shape after leakage removal: (44988, 14)

Columns:
['Age', 'Gender', 'Education', 'Person Income', 'Employee Experience', 'Home Ownership', 'Loan Amount', 'Loan Intent', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan Status', 'Loan_to_Income', 'Log_Person_Income']


Final data audit


In [45]:
print("Final dataset shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nTarget distribution:")
print(df_clean["Loan Status"].value_counts())

print("\nTarget percentages:")
print(
    df_clean["Loan Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Final dataset shape: (44988, 14)

Missing values:
Age                    0
Gender                 0
Education              0
Person Income          0
Employee Experience    0
Home Ownership         0
Loan Amount            0
Loan Intent            0
Credit History         0
Credit Score           0
Previous Loan          0
Loan Status            0
Loan_to_Income         0
Log_Person_Income      0
dtype: int64

Duplicate rows: 0

Target distribution:
Loan Status
0    34988
1    10000
Name: count, dtype: int64

Target percentages:
Loan Status
0    77.77
1    22.23
Name: proportion, dtype: float64


In [46]:
print("\nData types:")
print(df_clean.dtypes)


Data types:
Age                      int64
Gender                     str
Education                  str
Person Income            int64
Employee Experience      int64
Home Ownership             str
Loan Amount              int64
Loan Intent                str
Credit History           int64
Credit Score             int64
Previous Loan              str
Loan Status              int64
Loan_to_Income         float64
Log_Person_Income      float64
dtype: object


In [47]:
categorical_features = [
    "Gender",
    "Education",
    "Home Ownership",
    "Loan Intent",
    "Previous Loan"
]

numeric_features = [
    "Age",
    "Person Income",
    "Employee Experience",
    "Loan Amount",
    "Credit History",
    "Credit Score",
    "Loan_to_Income",
    "Log_Person_Income"
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total predictors:", len(numeric_features) + len(categorical_features))

Numeric features: 8
Categorical features: 5
Total predictors: 13


In [48]:
import os

os.makedirs("/workspaces/loan-approval-prediction/data/processed", exist_ok=True)

In [49]:
processed_path = "/workspaces/loan-approval-prediction/data/processed/loan_data_feature_engineered.csv"

df_clean.to_csv(processed_path, index=False)

print("Saved:", processed_path)
print("Shape:", df_clean.shape)

Saved: /workspaces/loan-approval-prediction/data/processed/loan_data_feature_engineered.csv
Shape: (44988, 14)
